#### Name: Blessing Adeniji
#### Degree: MSc Artifical Intelligence Online
#### Capstone Project: AI-Generated Text Detection - Deepfakes

##### Purpose: Fine-tune Ettin68m-Decorder on all 4 datasets and compare against the encorder of identical size.

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

# compute accuracy and F1 score for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # pick the class with the highest score
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

# Build a 4x4 Matrix
def save_results_to_csv(model_name, trained_on, tested_on, results):
    # Create a one-row table with the results
    row = pd.DataFrame({
        'model_name': [model_name],
        'trained_on': [trained_on],
        'tested_on': [tested_on],
        'accuracy': [results['eval_accuracy']],
        'f1': [results['eval_f1']],
        'loss': [results['eval_loss']]
    })

    # Append to results and create if it doesn't exist
    file_exists = os.path.isfile('all_models_evaluation_results.csv')
    row.to_csv('all_models_evaluation_results.csv', mode='a', header=not file_exists, index=False)
    print("Saved:", model_name, trained_on, tested_on)

In [ ]:
# Decoder run: Fine-tune on Abstracts

# Load Abstracts train/val splits
abstracts_train_dataset = pd.read_csv("data_splits/ChatGPT-Research-Abstracts_train.csv")
abstracts_validation_dataset = pd.read_csv("data_splits/ChatGPT-Research-Abstracts_val.csv")

# Load Decoder tokenizer
tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-decoder-68m")

# Decoders have no padding token by default, using end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
abstracts_train_ds = Dataset.from_pandas(abstracts_train_dataset).map(tokenize, batched=True)
abstracts_val_ds = Dataset.from_pandas(abstracts_validation_dataset).map(tokenize, batched=True)

# Load fresh DECODER with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-decoder-68m", num_labels=2)

# Decoder - this is the model's config for the token that is the pad token
model.config.pad_token_id = tokenizer.pad_token_id

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin_decoder68m_abstracts",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation 
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",           # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    bf16=True,                       # mixed precision - faster, less memory
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=abstracts_train_ds,              # training dataset
    eval_dataset=abstracts_val_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the Decoder model
trainer.train()

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/154 [00:00<?, ?it/s]

[transformers] ModernBertDecoderForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-decoder-68m
Key                  | Status     | 
---------------------+------------+-
decoder.bias         | UNEXPECTED | 
lm_head.dense.weight | UNEXPECTED | 
lm_head.norm.weight  | UNEXPECTED | 
decoder.weight       | UNEXPECTED | 
head.dense.weight    | MISSING    | 
classifier.weight    | MISSING    | 
head.norm.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.066422,0.042509,0.990667,0.990591
2,0.022559,0.036528,0.992667,0.992647
3,0.000008,0.045559,0.992333,0.992310


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2625, training_loss=0.0321947713182086, metrics={'train_runtime': 494.3502, 'train_samples_per_second': 84.96, 'train_steps_per_second': 5.31, 'total_flos': 5081803688116224.0, 'train_loss': 0.0321947713182086, 'epoch': 3.0})

In [ ]:
# Save the DECODER abstracts model and tokenizer
model.save_pretrained("models/decoder_abstracts_final")
tokenizer.save_pretrained("models/decoder_abstracts_final")

# Load and tokenize all 4 test sets with the Decoder tokenizer
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("ettin-decoder-68m", "ChatGPT-Abstracts", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-decoder-68m", "ChatGPT-Abstracts", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("ettin-decoder-68m", "ChatGPT-Abstracts", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-decoder-68m", "ChatGPT-Abstracts", "Mage", trainer.evaluate(mage_test_ds))

print("\nDecoder row 1 complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: ettin-decoder-68m ChatGPT-Abstracts RAID


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000008,1.598850,3,0.685444,0.542161


Saved: ettin-decoder-68m ChatGPT-Abstracts Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000008,4.229934,3,0.531076,0.128865


Saved: ettin-decoder-68m ChatGPT-Abstracts Mage

Decoder row 1 complete


In [7]:
# Decoder run: Fine-tune on wiki

# Load Wiki train/val splits
wiki_train_dataset = pd.read_csv("data_splits/GPT-Wiki-Intro_train.csv")
wiki_validation_dataset = pd.read_csv("data_splits/GPT-Wiki-Intro_val.csv")

# Load Decoder tokenizer
tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-decoder-68m")

# Decoders have no padding token by default, using end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
wiki_train_ds = Dataset.from_pandas(wiki_train_dataset).map(tokenize, batched=True)
wiki_val_ds = Dataset.from_pandas(wiki_validation_dataset).map(tokenize, batched=True)

# Load fresh DECODER with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-decoder-68m", num_labels=2)

# Decoder - this is the model's config for the token that is the pad token
model.config.pad_token_id = tokenizer.pad_token_id

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin_decoder68m_wiki",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation 
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",           # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    bf16=True,                       # mixed precision - faster, less memory
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=wiki_train_ds,         # training dataset
    eval_dataset=wiki_val_ds,            # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the Decoder model
trainer.train()

Map:   0%|          | 0/210000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/154 [00:00<?, ?it/s]

[transformers] ModernBertDecoderForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-decoder-68m
Key                  | Status     | 
---------------------+------------+-
decoder.bias         | UNEXPECTED | 
lm_head.dense.weight | UNEXPECTED | 
lm_head.norm.weight  | UNEXPECTED | 
decoder.weight       | UNEXPECTED | 
head.dense.weight    | MISSING    | 
classifier.weight    | MISSING    | 
head.norm.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.007337,0.001461,0.999667,0.999667
2,0.000003,0.002628,0.999533,0.999533
3,0.000000,0.002965,0.999667,0.999667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39375, training_loss=0.0031256777453044105, metrics={'train_runtime': 5713.5561, 'train_samples_per_second': 110.264, 'train_steps_per_second': 6.892, 'total_flos': 6.193603415438131e+16, 'train_loss': 0.0031256777453044105, 'epoch': 3.0})

In [ ]:
# Save the DECODER wiki model and tokenizer
model.save_pretrained("models/decoder_wiki_final")
tokenizer.save_pretrained("models/decoder_wiki_final")

# Load and tokenize all 4 test sets with the Decoder tokenizer
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("ettin-decoder-68m", "GPT-Wiki-Intro", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-decoder-68m", "GPT-Wiki-Intro", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-decoder-68m", "GPT-Wiki-Intro", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("ettin-decoder-68m", "GPT-Wiki-Intro", "Mage", trainer.evaluate(mage_test_ds))

print("\nDecoder row 2 complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: ettin-decoder-68m GPT-Wiki-Intro ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,2.003483,3,0.788578,0.762619


Saved: ettin-decoder-68m GPT-Wiki-Intro RAID


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000000,4.320720,3,0.534826,0.539872


Saved: ettin-decoder-68m GPT-Wiki-Intro Mage

Decoder row 2 complete


In [10]:
# Decoder run: Fine-tune on RAID

# Load RAID train/val splits
raid_train_dataset = pd.read_csv("data_splits/RAID_train.csv")
raid_validation_dataset = pd.read_csv("data_splits/RAID_val.csv")

# Load Decoder tokenizer
tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-decoder-68m")

# Decoders have no padding token by default, using end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
raid_train_ds = Dataset.from_pandas(raid_train_dataset).map(tokenize, batched=True)
raid_val_ds = Dataset.from_pandas(raid_validation_dataset).map(tokenize, batched=True)

# Load fresh DECODER with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-decoder-68m", num_labels=2)

# Decoder - this is the model's config for the token that is the pad token
model.config.pad_token_id = tokenizer.pad_token_id

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin_decoder68m_raid",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation 
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",           # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    bf16=True,                       # mixed precision - faster, less memory
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=raid_train_ds,              # training dataset
    eval_dataset=raid_val_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the Decoder model
trainer.train()

Map:   0%|          | 0/205429 [00:00<?, ? examples/s]

Map:   0%|          | 0/44020 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/154 [00:00<?, ?it/s]

[transformers] ModernBertDecoderForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-decoder-68m
Key                  | Status     | 
---------------------+------------+-
decoder.bias         | UNEXPECTED | 
lm_head.dense.weight | UNEXPECTED | 
lm_head.norm.weight  | UNEXPECTED | 
decoder.weight       | UNEXPECTED | 
head.dense.weight    | MISSING    | 
classifier.weight    | MISSING    | 
head.norm.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.110037,0.113228,0.971263,0.970698
2,0.039181,0.094930,0.978192,0.978021
3,0.031865,0.106244,0.981395,0.981327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=38520, training_loss=0.06990422016224386, metrics={'train_runtime': 7731.8582, 'train_samples_per_second': 79.707, 'train_steps_per_second': 4.982, 'total_flos': 8.065269319665254e+16, 'train_loss': 0.06990422016224386, 'epoch': 3.0})

In [11]:
# Save the DECODER RAID model and tokenizer
model.save_pretrained("models/decoder_raid_final")
tokenizer.save_pretrained("models/decoder_raid_final")

# Load and tokenize all 4 test sets with the Decoder tokenizer
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("ettin-decoder-68m", "RAID", "RAID", trainer.evaluate(raid_test_ds))
save_results_to_csv("ettin-decoder-68m", "RAID", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-decoder-68m", "RAID", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-decoder-68m", "RAID", "Mage", trainer.evaluate(mage_test_ds))

print("\nDecoder row 3 complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

...

Saved: ettin-decoder-68m RAID ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.031865,0.050257,3,0.989933,0.990011


Saved: ettin-decoder-68m RAID Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.031865,1.828004,3,0.724211,0.763370


Saved: ettin-decoder-68m RAID Mage

Decoder row 3 complete


In [12]:
# Decoder run: Fine-tune on MAGE

# Load MAGE train/val splits
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

# Load Decoder tokenizer
tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-decoder-68m")

# Decoders have no padding token by default, using end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert and tokenize
mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_val_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

# Load fresh DECODER with a 2-class classification head
model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-decoder-68m", num_labels=2)

# Decoder - this is the model's config for the token that is the pad token
model.config.pad_token_id = tokenizer.pad_token_id

# Training settings
training_args = TrainingArguments(
    output_dir="models/ettin_decoder68m_mage",  # output directory - where checkpoints and model will be saved
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=32,   # batch size for evaluation 
    learning_rate=2e-5,              # learning rate
    eval_strategy="epoch",           # evaluate each epoch
    save_strategy="epoch",           # save each epoch
    load_best_model_at_end=True,     # load the best model when finished training (default metric is loss)
    logging_steps=50,
    bf16=True,                       # mixed precision - faster, less memory
    report_to="none"
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=mage_train_ds,              # training dataset
    eval_dataset=mage_val_ds,          # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the Decoder model
trainer.train()

Map:   0%|          | 0/130645 [00:00<?, ? examples/s]

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/154 [00:00<?, ?it/s]

[transformers] ModernBertDecoderForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-decoder-68m
Key                  | Status     | 
---------------------+------------+-
decoder.bias         | UNEXPECTED | 
lm_head.dense.weight | UNEXPECTED | 
lm_head.norm.weight  | UNEXPECTED | 
decoder.weight       | UNEXPECTED | 
head.dense.weight    | MISSING    | 
classifier.weight    | MISSING    | 
head.norm.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.182656,0.123995,0.953956,0.953698
2,0.100248,0.172395,0.961815,0.961883
3,0.032947,0.224793,0.963672,0.963419


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24498, training_loss=0.09942759343050618, metrics={'train_runtime': 4760.4429, 'train_samples_per_second': 82.332, 'train_steps_per_second': 5.146, 'total_flos': 5.031747659317862e+16, 'train_loss': 0.09942759343050618, 'epoch': 3.0})

In [13]:
# Save the DECODER MAGE model and tokenizer
model.save_pretrained("models/decoder_mage_final")
tokenizer.save_pretrained("models/decoder_mage_final")

# Load and tokenize all 4 test sets with the Decoder tokenizer
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("ettin-decoder-68m", "MAGE", "Mage", trainer.evaluate(mage_test_ds))
save_results_to_csv("ettin-decoder-68m", "MAGE", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-decoder-68m", "MAGE", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-decoder-68m", "MAGE", "RAID", trainer.evaluate(raid_test_ds))


print("\nDecoder row 4 complete - 4x4 Matrix Complete - Core comparison done")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

...

Saved: ettin-decoder-68m MAGE ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.032947,0.320491,3,0.880844,0.884767


Saved: ettin-decoder-68m MAGE Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.032947,0.936164,3,0.780559,0.801944


Saved: ettin-decoder-68m MAGE RAID

Decoder row 4 complete - 4x4 Matrix Complete - Core comparison done
